# AITH DL Competition – Tabular (2025)

Бинарная классификация `smoking` только нейросетями. Ноутбук строит безопасный с точки зрения утечек конвейер: кросс-валидация, препроцессинг по фолдам, MLP, ранняя остановка по ROC AUC и генерация сабмишена.

## Гарантии корректности
- Целевой столбец `smoking` удаляется до любых преобразований.
- StratifiedKFold(shuffle=True, random_state=SEED) + OOF-предсказания для честного CV.
- Импьютация медианой и скейлинг подгоняются **только** на трейне фолда; на валидацию/тест лишь применяются.
- Тест не участвует в обучении/ранней остановке.
- Фиксированные сиды + ограничение потоков для воспроизводимости.
- Запрещённые модели (деревья/бустинг) не используются — чистый MLP на Torch.

In [29]:
#!pip install pandas scikit-learn

In [30]:

import os
import random
from pathlib import Path
from typing import List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Reproducibility
SEED = 42
os.environ.setdefault("PYTHONHASHSEED", str(SEED))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
torch.set_num_threads(1)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


device(type='cpu')

In [31]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo


def get_project_root() -> Path:
    """Locate repo root containing data/competition starting from cwd upwards."""
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "data" / "competition" / "train.csv").exists():
            return base
    raise FileNotFoundError("data/competition/train.csv not found from current working directory.")

PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "competition"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "smoking"
ID_COLS = ["id"]

# Timestamped submission name in Moscow time
ts_msk = datetime.now(ZoneInfo("Europe/Moscow")).strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = RESULTS_DIR / f"submission_{ts_msk}.csv"

# Model/training hyperparameters
HIDDEN_DIMS = (256, 128, 64)
DROPOUT = 0.2
BATCH_SIZE = 256
MAX_EPOCHS = 100
PATIENCE = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4
N_SPLITS = 5


## Определение типов признаков
Автоматически делим столбцы на категориальные и числовые. Целочисленные признаки с малой мощностью (<20 уникальных значений) считаем категориальными (на случай эмбеддингов/кодировок); остальные — числовые.

In [32]:

from pandas.api.types import is_integer_dtype


def detect_feature_types(df: pd.DataFrame, target_col: str, id_cols: Sequence[str]) -> Tuple[List[str], List[str]]:
    feature_cols = [c for c in df.columns if c not in set(id_cols) | {target_col}]
    cat_cols: List[str] = []
    num_cols: List[str] = []
    for col in feature_cols:
        dtype = df[col].dtype
        if dtype == "object" or str(dtype).startswith("category"):
            cat_cols.append(col)
        elif is_integer_dtype(dtype) and df[col].nunique() < 20:
            cat_cols.append(col)
        else:
            num_cols.append(col)
    return cat_cols, num_cols


## Препроцессинг по фолдам
- Числовые: импьютация медианой + StandardScaler.
- Категориальные: OrdinalEncoder с маркером unknown.
Все трансформеры фитятся только на трейн-сплите конкретного фолда, чтобы исключить утечки.

In [33]:

class FoldPreprocessor:
    def __init__(self, cat_cols: Sequence[str], num_cols: Sequence[str]) -> None:
        self.cat_cols = list(cat_cols)
        self.num_cols = list(num_cols)
        self.num_impute_: Optional[pd.Series] = None
        self.scaler = StandardScaler()
        self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1) if self.cat_cols else None

    def fit(self, df: pd.DataFrame) -> "FoldPreprocessor":
        if self.num_cols:
            num_df = df[self.num_cols]
            self.num_impute_ = num_df.median()
            num_filled = num_df.fillna(self.num_impute_)
            self.scaler.fit(num_filled)
        if self.cat_cols and self.encoder is not None:
            cat_df = df[self.cat_cols].astype("object").fillna("missing")
            self.encoder.fit(cat_df)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        parts: List[np.ndarray] = []
        if self.num_cols:
            assert self.num_impute_ is not None, "Numeric imputer not fitted."
            num_df = df[self.num_cols].fillna(self.num_impute_)
            parts.append(self.scaler.transform(num_df))
        if self.cat_cols and self.encoder is not None:
            cat_df = df[self.cat_cols].astype("object").fillna("missing")
            parts.append(self.encoder.transform(cat_df))
        if not parts:
            raise ValueError("No features available after preprocessing.")
        return np.hstack(parts).astype(np.float32)


## Модель и мотивация
MLP с BatchNorm и Dropout: простой, устойчивый к шуму, подходит табличке без деревьев. AdamW даёт стабильную оптимизацию с weight decay. Слои (256,128,64) — умеренная ёмкость для 20–30 фич без переобучения, Dropout=0.2 для регуляризации.

In [34]:

class TabularDataset(Dataset):
    def __init__(self, features: np.ndarray, targets: Optional[np.ndarray] = None) -> None:
        self.features = torch.from_numpy(features).float()
        self.targets = None if targets is None else torch.from_numpy(targets.astype(np.float32))

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, idx: int):
        if self.targets is None:
            return self.features[idx]
        return self.features[idx], self.targets[idx]


class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: Sequence[int], dropout: float = 0.2) -> None:
        super().__init__()
        layers: List[nn.Module] = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


## Обучение
- BCEWithLogitsLoss для бинарной задачи.
- AdamW (lr=1e-3, weight_decay=1e-4) — баланс скорости/стабильности.
- Ранняя остановка по валидационному ROC AUC (patience=10) — не даём переобучаться, метрика соответствует цели соревнования.

In [35]:

def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0, generator=generator)


def train_one_fold(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    max_epochs: int,
    patience: int,
    lr: float,
    weight_decay: float,
) -> Tuple[dict, float]:
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_auc = -np.inf
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_preds: List[np.ndarray] = []
        val_targets: List[np.ndarray] = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                logits = model(xb)
                probs = torch.sigmoid(logits).cpu().numpy()
                val_preds.append(probs)
                val_targets.append(yb.numpy())

        val_pred = np.concatenate(val_preds)
        val_y = np.concatenate(val_targets)
        val_auc = roc_auc_score(val_y, val_pred)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    assert best_state is not None, "Training did not run."
    return best_state, best_auc


def predict_proba(model: nn.Module, loader: DataLoader, device: torch.device) -> np.ndarray:
    model.eval()
    preds: List[np.ndarray] = []
    with torch.no_grad():
        for batch in loader:
            xb = batch[0] if isinstance(batch, (list, tuple)) else batch
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds.append(probs)
    return np.concatenate(preds)


## CV-пайплайн
Stratified K-fold, на каждом фолде свой препроцессинг и модель. OOF-предсказания для честной метрики; тестовые предсказания усредняются по фолдам.

In [36]:
def run_cv_pipeline(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    n_splits: int = 5,
    hidden_dims: Sequence[int] = (128, 64),
    dropout: float = 0.2,
    batch_size: int = 256,
    max_epochs: int = 100,
    patience: int = 10,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
) -> Tuple[np.ndarray, np.ndarray, List[float]]:
    y = train_df[TARGET_COL].values.astype(np.float32)
    cat_cols, num_cols = detect_feature_types(train_df, TARGET_COL, ID_COLS)
    print(f"Detected {len(num_cols)} numerical and {len(cat_cols)} categorical features.")
    if not num_cols and not cat_cols:
        raise ValueError("No features found for training.")

    oof_pred = np.zeros(len(train_df), dtype=np.float32)
    test_pred = np.zeros(len(test_df), dtype=np.float32)
    fold_aucs: List[float] = []

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y), start=1):
        print(f"\nFold {fold}/{n_splits}")
        tr_df = train_df.iloc[train_idx].reset_index(drop=True)
        va_df = train_df.iloc[val_idx].reset_index(drop=True)

        preprocessor = FoldPreprocessor(cat_cols=cat_cols, num_cols=num_cols)
        preprocessor.fit(tr_df)

        X_tr = preprocessor.transform(tr_df)
        X_va = preprocessor.transform(va_df)
        X_te = preprocessor.transform(test_df)

        y_tr = tr_df[TARGET_COL].values.astype(np.float32)
        y_va = va_df[TARGET_COL].values.astype(np.float32)

        train_ds = TabularDataset(X_tr, y_tr)
        val_ds = TabularDataset(X_va, y_va)
        test_ds = TabularDataset(X_te)

        train_loader = make_loader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = make_loader(val_ds, batch_size=batch_size, shuffle=False)
        test_loader = make_loader(test_ds, batch_size=batch_size, shuffle=False)

        model = MLP(input_dim=X_tr.shape[1], hidden_dims=hidden_dims, dropout=dropout).to(DEVICE)

        best_state, best_auc = train_one_fold(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=DEVICE,
            max_epochs=max_epochs,
            patience=patience,
            lr=lr,
            weight_decay=weight_decay,
        )
        fold_aucs.append(best_auc)
        print(f"Fold {fold} best ROC AUC: {best_auc:.5f}")

        model.load_state_dict(best_state)

        oof_pred[val_idx] = predict_proba(model, val_loader, DEVICE)
        test_pred += predict_proba(model, test_loader, DEVICE) / n_splits

    cv_auc = roc_auc_score(y, oof_pred)
    print(f"\nOOF ROC AUC: {cv_auc:.5f}")
    return oof_pred, test_pred, fold_aucs


## Загрузка данных и запуск CV

In [37]:

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

# Align test to sample submission order
test_df = test_df.merge(sample_sub[["id"]], on="id", how="right")

feature_train_df = train_df.drop(columns=ID_COLS)
feature_test_df = test_df.drop(columns=ID_COLS)

_, test_pred, fold_aucs = run_cv_pipeline(
    train_df=feature_train_df,
    test_df=feature_test_df,
    n_splits=N_SPLITS,
    hidden_dims=HIDDEN_DIMS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

print("Fold ROC AUCs:", [f"{auc:.5f}" for auc in fold_aucs])
print(f"Mean ROC AUC: {np.mean(fold_aucs):.5f}")


Detected 22 numerical and 0 categorical features.

Fold 1/5
Fold 1 best ROC AUC: 0.87729

Fold 2/5
Fold 2 best ROC AUC: 0.88652

Fold 3/5
Fold 3 best ROC AUC: 0.88519

Fold 4/5
Fold 4 best ROC AUC: 0.88949

Fold 5/5
Fold 5 best ROC AUC: 0.88336

OOF ROC AUC: 0.88157
Fold ROC AUCs: ['0.87729', '0.88652', '0.88519', '0.88949', '0.88336']
Mean ROC AUC: 0.88437


## Формирование submission

In [38]:

submission = sample_sub.copy()
submission["smoking"] = np.clip(test_pred, 0.0, 1.0)
submission.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH


PosixPath('/Users/savely/aihub/deep_learning/dl_hw/results/submission_20260124_233607.csv')